In [41]:
import pandas as pd
import numpy as np

In [42]:
# Seed para reprodutibilidade
np.random.seed(8)

# Segmentação da base de interações entre treino e teste

Como não é um processo interativo, não é necessário separar dados para validação.

Para cada usuário, todas as interações de algum produto são removidos.

Como as recomendações são com base em itens não adquidos, o ideal é remover apenas o item que foi menos adquirido pelo usuário.

In [43]:
interacoes = pd.read_parquet('../dados_sinteticos/itens.parquet')

In [44]:
interacoes

,id_produto,id_usuario,deslocamento
0,P_0,U_958,4760.773531
1,P_0,U_981,6289.107176
2,P_0,U_16,2816.495495
3,P_0,U_454,1602.160816
4,P_0,U_208,6332.289519
...,...,...,...
9995,P_99,U_833,2057.534783
9996,P_99,U_609,5246.819559
9997,P_99,U_125,2273.763022
9998,P_99,U_495,5037.035703


In [45]:
# Quantos produtos distintos cada usuário adquiriu
produtos_por_usuario = interacoes.groupby('id_usuario')['id_produto'].nunique()

# Usuários com apenas 1 produto distinto não podem entrar no split
usuarios_avaliáveis = produtos_por_usuario[produtos_por_usuario >= 2].index
print(f"Usuários avaliáveis (≥2 produtos distintos): {len(usuarios_avaliáveis)}")

usuarios_nao_avaliáveis = produtos_por_usuario[produtos_por_usuario < 2].index
print(f"Usuários excluídos da avaliação (1 produto): {len(usuarios_nao_avaliáveis)}")

Usuários avaliáveis (≥2 produtos distintos): 903
Usuários excluídos da avaliação (1 produto): 74


In [47]:
# split removendo um item por usuário

# Produtos que aparecem em pelo menos 2 usuários distintos
usuarios_por_produto = interacoes.groupby('id_produto')['id_usuario'].nunique()
produtos_sorteáveis = set(usuarios_por_produto[usuarios_por_produto >= 2].index)

linhas_treino = []
linhas_teste  = []

for id_usuario, grupo in interacoes.groupby('id_usuario'):

    # Usuários com 1 produto distinto vão inteiros pro treino
    if id_usuario not in usuarios_avaliáveis:
        linhas_treino.append(grupo)
        continue

    # Produtos distintos desse usuário
    produtos_distintos = grupo['id_produto'].unique()

    # Só sorteia entre produtos que são seguros de remover
    candidatos_teste = [p for p in produtos_distintos if p in produtos_sorteáveis]

    # Se não houver candidato seguro, o usuário vai inteiro pro treino
    if not candidatos_teste:
        linhas_treino.append(grupo)
        continue

    produto_teste = np.random.choice(candidatos_teste)

    # Todas as linhas desse produto para esse usuário vão pro teste
    mask_teste = grupo['id_produto'] == produto_teste
    linhas_teste.append(grupo[mask_teste])
    linhas_treino.append(grupo[~mask_teste])

interacoes_treino = pd.concat(linhas_treino).reset_index(drop=True)
interacoes_teste  = pd.concat(linhas_teste).reset_index(drop=True)

In [48]:
print(f"Treino: {len(interacoes_treino)} linhas")
print(f"Teste:  {len(interacoes_teste)} linhas")

Treino: 7935 linhas
Teste:  2065 linhas


In [49]:
#  Verificação do split

# Nenhum usuário deve estar no teste sem estar no treino
# os usuários no teste precisam formar um subset dos usuários em treino
usuarios_treino = set(interacoes_treino['id_usuario'].unique())
usuarios_teste  = set(interacoes_teste['id_usuario'].unique())
assert usuarios_teste.issubset(usuarios_treino), "Há usuários no teste sem histórico no treino!"

# Nenhum par usuário-produto deve estar em ambos
# um par de usuário-produto não pode existir tanto no treino quanto no teste
# necessário porque a recomendação se baseia em itens não adquiridos
pares_treino = set(zip(interacoes_treino['id_usuario'], interacoes_treino['id_produto']))
pares_teste  = set(zip(interacoes_teste['id_usuario'],  interacoes_teste['id_produto']))
assert pares_treino.isdisjoint(pares_teste), "Há vazamento de pares usuário-produto!"

# Verificação de persistência de produtos
produtos_originais = set(interacoes['id_produto'].unique())
produtos_treino    = set(interacoes_treino['id_produto'].unique())
produtos_perdidos  = produtos_originais - produtos_treino

if produtos_perdidos:
    print(f"{len(produtos_perdidos)} produto(s) sumiram do treino: {produtos_perdidos}")
else:
    print("Todos os produtos persistem no treino.")

print(f"Split validado: {len(interacoes_treino)} linhas treino | {len(interacoes_teste)} linhas teste")

Todos os produtos persistem no treino.
Split validado: 7935 linhas treino | 2065 linhas teste


In [50]:
interacoes_treino.to_parquet('../dados_sinteticos/interacoes_treino.parquet', index=False, engine='pyarrow', compression='snappy')
interacoes_teste.to_parquet('../dados_sinteticos/interacoes_teste.parquet', index=False, engine='pyarrow', compression='snappy')